# 读取和清理不同格式的资料

本页从 CSV、DOCX、PPTX fixture 读取真实文件，统一为 `text + metadata`；解析器只做本地工作，不调用外部模型。

In [1]:
from pathlib import Path
import csv
from docx import Document
from pptx import Presentation

def find_c7_root(start):
    start = Path(start).resolve()
    for folder in (start, *start.parents):
        if (folder / 'data' / 'dataset/manifest.json').is_file(): return folder
        nested = folder / 'notebook' / 'C7 高级 RAG 技巧'
        if (nested / 'data' / 'dataset/manifest.json').is_file(): return nested
    raise FileNotFoundError('请从仓库根、C7 根或本章目录启动')

c7_root = find_c7_root(Path.cwd())
fixture_root = c7_root / 'data' / 'fixtures'
csv_records = []
with (fixture_root / 'company_sample.csv').open(encoding='utf-8', newline='') as handle:
    for row_number, row in enumerate(csv.DictReader(handle), start=2):
        body = '；'.join(f'{key}：{value}' for key, value in row.items())
        csv_records.append({'text': body, 'metadata': {'source': 'company_sample.csv', 'row': row_number}})

docx_path = c7_root / '2. 数据处理' / 'data' / '1. 简介 Introduction.docx'
docx_records = [
    {'text': paragraph.text.strip(), 'metadata': {'source': docx_path.name, 'paragraph': number}}
    for number, paragraph in enumerate(Document(docx_path).paragraphs, 1)
    if paragraph.text.strip()
]

pptx_path = c7_root / '2. 数据处理' / 'data' / 'AI视频.pptx'
pptx_records = []
for slide_number, slide in enumerate(Presentation(pptx_path).slides, 1):
    text = '\n'.join(
        shape.text.strip()
        for shape in slide.shapes
        if hasattr(shape, 'text') and shape.text.strip()
    )
    if text:
        pptx_records.append({'text': text, 'metadata': {'source': pptx_path.name, 'slide': slide_number}})

records = csv_records + docx_records + pptx_records
assert csv_records and docx_records and pptx_records
assert all(item['text'] and item['metadata']['source'] for item in records)
print(f'解析记录数：CSV={len(csv_records)}，DOCX={len(docx_records)}，PPTX={len(pptx_records)}，合计={len(records)}')
print(records[0])

解析记录数：CSV=3，DOCX=8，PPTX=14，合计=25
{'text': 'company：青松科技；team：搜索；summary：维护公开资料检索流程', 'metadata': {'source': 'company_sample.csv', 'row': 2}}


## 清理重复文字和敏感信息

清理只去除重复行、连续空白和明确要求脱敏的联系方式；保留标题、标点和换行，避免破坏文档/幻灯片边界。

In [2]:
import re
def clean_text(text, redact_contacts=True):
    lines = [re.sub(r'[^\S\n]+', ' ', line).strip() for line in text.splitlines()]
    value = '\n'.join(dict.fromkeys(line for line in lines if line))
    return re.sub(r'[\w.+-]+@[\w.-]+', '[邮箱已隐去]', value) if redact_contacts else value
raw = '项目手册  联系 owner@example.com\n检索前先检查资料权限。\n项目手册  联系 owner@example.com'
cleaned = clean_text(raw)
print(cleaned)
assert cleaned.count('项目手册') == 1 and 'owner@example.com' not in cleaned

项目手册 联系 [邮箱已隐去]
检索前先检查资料权限。


## 结构边界检查

CSV 用行号定位，DOCX 用段落号定位，PPTX 用页码定位；后续分块可据此回溯原始资料。

## 解析后抽查与限制

Word 的标题层级和表格不能依赖 `paragraphs` 自动保留；PPT 图片文字需要 OCR，并检查文本框阅读顺序；PDF 扫描页和双栏阅读顺序也要抽查。清理应保留标题、单位、标点和结构边界。

保留不可变原始副本，另生成检索副本做脱敏，不在原始资料上就地清理。本页直接依赖 `python-docx` 和 `python-pptx`；缺少依赖时应按 `requirements-c7.txt` 安装，不降级、不跳过对应格式。